<a href="https://colab.research.google.com/github/BioML-UGent/Advanced-AI-for-Bioinformatics/blob/main/03_CNNs/03_CNNs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PC lab 3: Convolutional neural networks

An MLP connects every input to every hidden unit, so it has no idea which pixels are next to each other. A **convolutional neural network** (CNN) looks for small, local patterns, with the same pattern detector at every position. In **part 1**, we apply CNNs to images; in **part 2**, to DNA sequences.

**Learning goals**
- Understand what a convolution computes, and how padding, stride and pooling change the output shape
- Build and train a CNN in PyTorch
- Apply 1D convolutions to one-hot encoded DNA to predict splice sites, and interpret the learned filters

## 0 Setup

The cells below load MNIST and the training functions from PC lab 2. Note that we keep the images as images: the data has shape (number of images, channels, height, width), with 1 channel for grayscale images.

In [ ]:
import copy
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
from torchvision import datasets

torch.manual_seed(42)
np.random.seed(42)

# MNIST, as images of shape (1, 28, 28)
train_data = datasets.MNIST(root="data", train=True, download=True)
X_train_mnist = train_data.data.float().unsqueeze(1) / 255
y_train_mnist = train_data.targets

perm = torch.randperm(len(X_train_mnist))
n_train = int(0.8 * len(X_train_mnist))
X_val_mnist, y_val_mnist = X_train_mnist[perm[n_train:]], y_train_mnist[perm[n_train:]]
X_train_mnist, y_train_mnist = X_train_mnist[perm[:n_train]], y_train_mnist[perm[:n_train]]

train_loader = DataLoader(TensorDataset(X_train_mnist, y_train_mnist), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val_mnist, y_val_mnist), batch_size=1000, shuffle=False)

print(X_train_mnist.shape, X_val_mnist.shape)

In [ ]:
# Training functions from PC lab 2
def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    batch_losses = []
    for X_batch, y_batch in loader:
        optimizer.zero_grad()
        loss = loss_fn(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        batch_losses.append(loss.item())
    return batch_losses


def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for X_batch, y_batch in loader:
            logits = model(X_batch)
            total_loss += loss_fn(logits, y_batch).item() * len(y_batch)
            correct += (logits.argmax(dim=1) == y_batch).sum().item()
            n += len(y_batch)
    return total_loss / n, correct / n


def fit(model, optimizer, train_loader, val_loader, n_epochs, verbose=True):
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    for epoch in range(1, n_epochs + 1):
        batch_losses = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(np.mean(batch_losses))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if verbose:
            print(f"epoch {epoch}: train loss {history['train_loss'][-1]:.3f}, "
                  f"val loss {val_loss:.3f}, val accuracy {val_acc:.3f}")
    return history


def fit_early_stopping(model, optimizer, train_loader, val_loader, n_epochs, patience):
    """Train until the validation loss has not improved for `patience` epochs; restore the best weights."""
    loss_fn = nn.CrossEntropyLoss()
    history = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_loss, best_epoch, best_state = float("inf"), 0, None
    for epoch in range(1, n_epochs + 1):
        batch_losses = train_one_epoch(model, train_loader, loss_fn, optimizer)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn)
        history["train_loss"].append(np.mean(batch_losses))
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)
        if val_loss < best_val_loss:
            best_val_loss, best_epoch = val_loss, epoch
            best_state = copy.deepcopy(model.state_dict())
        elif epoch - best_epoch >= patience:
            break
    model.load_state_dict(best_state)
    history["best_epoch"] = best_epoch
    return history


def plot_history(history, title=""):
    """Plot the training and validation loss per epoch, and mark the epoch with the best validation loss."""
    epochs = np.arange(1, len(history["train_loss"]) + 1)
    plt.plot(epochs, history["train_loss"], label="training loss")
    plt.plot(epochs, history["val_loss"], label="validation loss")
    plt.axvline(history["best_epoch"], color="gray", linestyle="--", label="best epoch")
    plt.xlabel("epoch")
    plt.title(title)
    plt.legend()
    plt.show()


def n_parameters(model):
    return sum(p.numel() for p in model.parameters())

# Part 1: CNNs for images

## 1 The convolution

A convolutional layer slides a small **kernel** (for example 3 × 3 weights) over the image. At every position, it multiplies the kernel element-wise with the image patch under it and sums the result. The outputs together form a new image: the **feature map**. The **same** kernel is used at every position. Below, we apply two hand-made kernels with `F.conv2d`; in a CNN, the kernels are learned.

In [ ]:
image = X_train_mnist[:1]  # shape (1, 1, 28, 28): one image with one channel

kernels = torch.tensor([[[[-1., 0., 1.],
                          [-2., 0., 2.],
                          [-1., 0., 1.]]],
                        [[[-1., -2., -1.],
                          [ 0.,  0.,  0.],
                          [ 1.,  2.,  1.]]]])  # shape (2 kernels, 1 input channel, 3, 3)

feature_maps = F.conv2d(image, kernels)
print("input:", image.shape, "-> output:", feature_maps.shape)

fig, axes = plt.subplots(1, 3, figsize=(10, 3.5))
axes[0].imshow(image[0, 0], cmap="gray")
axes[0].set_title("input image")
for ax, fmap, title in zip(axes[1:], feature_maps[0], ["kernel 1", "kernel 2"]):
    ax.imshow(fmap, cmap="RdBu_r", vmin=-fmap.abs().max(), vmax=fmap.abs().max())
    ax.set_title(f"feature map of {title}")
for ax in axes:
    ax.axis("off")
plt.show()

<div class="alert alert-success">

<b>EXERCISE 1 (THINK): what does a kernel detect?</b>

a) What kind of pattern does each kernel detect? (Red: large positive output, blue: large negative output.)

b) Why are the feature maps 26 × 26 instead of 28 × 28?

</div>

**Your answer:**

## 2 Padding, stride and pooling

- **Padding** $P$ adds a border of zeros, so that the output can keep the size of the input.
- **Stride** $S$ is the step size of the kernel; a stride of 2 halves the height and width.
- **Max pooling** (`nn.MaxPool2d`) keeps the maximum of every window, e.g. 2 × 2. It reduces the size, and makes the network less sensitive to small shifts.

For an input of width $W$ and a kernel of size $K$: $W_\text{out} = \lfloor (W + 2P - K)/S \rfloor + 1$. A layer `nn.Conv2d(in_channels, out_channels, kernel_size)` has `out_channels` kernels, each of size `in_channels × kernel_size × kernel_size`, and every kernel produces one output channel.

<div class="alert alert-success">

<b>EXERCISE 2 (THINK): shapes and parameters</b>

a) The input `x` has shape `(8, 1, 28, 28)`. Predict the output shapes of the four layers in the cell below, then run it to check.

b) The cell also compares the number of parameters of `nn.Conv2d(1, 16, 3)` with a linear layer that produces the same number of outputs (16 × 26 × 26) from the 784 pixels. Why does the convolutional layer need so few?

</div>

In [ ]:
x = torch.randn(8, 1, 28, 28)
layers = {
    "nn.Conv2d(1, 16, 3)": nn.Conv2d(1, 16, 3),
    "nn.Conv2d(1, 16, 3, padding=1)": nn.Conv2d(1, 16, 3, padding=1),
    "nn.Conv2d(1, 16, 3, stride=2, padding=1)": nn.Conv2d(1, 16, 3, stride=2, padding=1),
    "nn.MaxPool2d(2)": nn.MaxPool2d(2),
}
for name, layer in layers.items():
    print(f"{name:42s} -> {tuple(layer(x).shape)}")

print()
print("parameters of nn.Conv2d(1, 16, 3):         ", n_parameters(nn.Conv2d(1, 16, 3)))
print("parameters of nn.Linear(784, 16 * 26 * 26):", n_parameters(nn.Linear(784, 16 * 26 * 26)))

**Your answer:**

## 3 A CNN for MNIST

A typical CNN for classification has a **convolutional part**, with blocks of convolution → ReLU → pooling (the height and width shrink, the number of channels grows), followed by a **classifier**: the feature maps are flattened (`nn.Flatten`) and passed to a linear layer that outputs the logits.

<div class="alert alert-success">

<b>EXERCISE 3: a CNN for MNIST</b>

a) Complete `SmallCNN` below:
- `self.features`: two blocks of `nn.Conv2d` (kernel size 3, padding 1) → `nn.ReLU` → `nn.MaxPool2d(2)`, with 16 channels in the first block and 32 in the second,
- `self.classifier`: `nn.Flatten()` and an `nn.Linear` to the 10 classes. Work out its number of inputs from the shape after `self.features`.

b) Train a `SmallCNN` with `fit` for 2 epochs, with Adam and learning rate 0.001. For comparison: the MLP of PC lab 2 ($784 \to 128 \to 10$, 101,770 parameters) reached about 94% after one epoch with Adam.

</div>

a)

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        ######## YOUR CODE HERE #########
        self.features = ...    # two blocks: Conv2d -> ReLU -> MaxPool2d (16 and 32 channels)
        self.classifier = ...  # Flatten -> Linear to the 10 classes
        #################################

    def forward(self, x):
        return self.classifier(self.features(x))


model = SmallCNN()
print("after the convolutional part:", model.features(x).shape)
print("output:", model(x).shape)
print("number of parameters:", n_parameters(model))

b)

In [ ]:
######## YOUR CODE HERE #########
torch.manual_seed(0)
cnn = ...        # a new SmallCNN
optimizer = ...  # Adam with learning rate 0.001
history = ...    # train for 2 epochs with fit
#################################

With 5 times fewer parameters than the MLP, the CNN reaches about 98%.

## 4 CNN vs MLP on a small, noisy dataset

<div class="alert alert-success">

<b>EXERCISE 4 (THINK): why do CNNs work so well?</b>

In PC lab 2, a large MLP ($784 \to 512 \to 512 \to 10$) quickly overfitted on 1000 training images with 20% wrong labels. The cell below repeats this experiment for the MLP and for our `SmallCNN`, both with Adam and early stopping (this takes about a minute). The CNN is clearly better, with about 30 times fewer parameters. Why?

</div>

In [ ]:
torch.manual_seed(0)
X_small, y_small = X_train_mnist[:1000], y_train_mnist[:1000].clone()
noisy = torch.rand(len(y_small)) < 0.2
y_small[noisy] = torch.randint(0, 10, (int(noisy.sum()),))  # 20% random labels
small_train_loader = DataLoader(TensorDataset(X_small, y_small), batch_size=32, shuffle=True)
small_val_loader = DataLoader(TensorDataset(X_val_mnist[:2000], y_val_mnist[:2000]), batch_size=1000, shuffle=False)

torch.manual_seed(0)
mlp = nn.Sequential(nn.Flatten(), nn.Linear(784, 512), nn.ReLU(), nn.Linear(512, 512), nn.ReLU(), nn.Linear(512, 10))
torch.manual_seed(0)
small_cnn = SmallCNN()

for name, model in [("MLP", mlp), ("CNN", small_cnn)]:
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    fit_early_stopping(model, optimizer, small_train_loader, small_val_loader, n_epochs=50, patience=5)
    val_loss, val_acc = evaluate(model, small_val_loader, nn.CrossEntropyLoss())
    print(f"{name}: {n_parameters(model):7d} parameters, validation accuracy {val_acc:.3f}")

**Your answer:**

---

# Part 2: CNNs for DNA sequences

## 5 Predicting splice sites

During splicing, the introns are cut out of a pre-mRNA. The boundaries are recognized by sequence signals: almost every intron starts with **GT** (the **donor** site, at the exon → intron boundary) and ends with **AG** (the **acceptor** site, at the intron → exon boundary). But GT and AG occur everywhere in the genome, so the surrounding bases determine whether a GT or AG is a real splice site. We will train a CNN to recognize real splice sites.

### One-hot encoding

A CNN needs numbers, not letters. With **one-hot encoding**, every position of a DNA sequence becomes a vector of 4 values (for A, C, G and T), with a 1 for the base at that position and 0 for the others. A sequence of length $L$ becomes a tensor of shape $(4, L)$: a one-dimensional "image" with 4 channels, one per base. The function below does this; ambiguous bases (such as N, "any base") get 0.25 for every base. The printed example is the encoding of ACGTTA: one row per base (A, C, G, T), one column per position.

On such a tensor, a **1D convolution** (`nn.Conv1d`) slides a kernel of shape $(4, k)$ along the sequence. The kernel has a weight for every base at every one of its $k$ positions, so it scores how well a stretch of $k$ bases matches a pattern: it is a motif detector, whose weights are learned during training.

In [ ]:
def one_hot(sequences):
    """One-hot encode a list of DNA sequences of equal length: shape (n_sequences, 4, length).
    Ambiguous bases (such as N) get 0.25 for every base."""
    index = {"A": 0, "C": 1, "G": 2, "T": 3}
    X = torch.zeros(len(sequences), 4, len(sequences[0]))
    for i, sequence in enumerate(sequences):
        for j, base in enumerate(sequence):
            if base in index:
                X[i, index[base], j] = 1
            else:
                X[i, :, j] = 0.25
    return X


print(one_hot(["ACGTTA"])[0])

### The dataset

We use the [UCI splice-junction dataset](https://archive.ics.uci.edu/dataset/69/molecular+biology+splice+junction+gene+sequences): 3190 human sequences of 60 bases, centered around a possible splice site (between positions 29 and 30, counting from 0). There are three classes: donor (EI, exon → intron), acceptor (IE, intron → exon), and neither (N). The cell below downloads the data, one-hot encodes the sequences, splits them into a training and a validation set, and prints an example of every class, with a | at the center.

In [ ]:
import io
import urllib.request
import zipfile

url = "https://archive.ics.uci.edu/static/public/69/molecular+biology+splice+junction+gene+sequences.zip"
with zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())) as z:
    lines = z.read("splice.data").decode().strip().split("\n")

class_names = ["donor", "acceptor", "neither"]
class_index = {"EI": 0, "IE": 1, "N": 2}
sequences = [line.split(",")[2].strip() for line in lines]
y_splice = torch.tensor([class_index[line.split(",")[0].strip()] for line in lines])
X_splice = one_hot(sequences)

perm = torch.randperm(len(X_splice), generator=torch.Generator().manual_seed(0))  # fixed split
train_idx, val_idx = perm[:2400], perm[2400:]
splice_train_loader = DataLoader(TensorDataset(X_splice[train_idx], y_splice[train_idx]), batch_size=64, shuffle=True)
splice_val_loader = DataLoader(TensorDataset(X_splice[val_idx], y_splice[val_idx]), batch_size=1000, shuffle=False)

print(X_splice.shape, "classes:", torch.bincount(y_splice))
for c in range(3):
    i = (y_splice == c).nonzero()[0].item()
    print(f"{class_names[c]:8s}: {sequences[i][:30]} | {sequences[i][30:]}")

Before training a CNN, it is always a good idea to try a simple baseline. The rule below only looks at the two bases at the center: GT at positions 30 and 31 means a donor, AG at positions 28 and 29 an acceptor, and anything else neither.

In [ ]:
def simple_rule(sequence):
    if sequence[30:32] == "GT":
        return 0  # donor
    if sequence[28:30] == "AG":
        return 1  # acceptor
    return 2      # neither


rule_predictions = torch.tensor([simple_rule(sequences[i]) for i in val_idx])
print(f"accuracy of the simple rule on the validation set: {(rule_predictions == y_splice[val_idx]).float().mean():.3f}")

donor_windows = [sequences[i][27:36] for i in range(len(sequences)) if y_splice[i] == 0]
decoys = sum(simple_rule(sequences[i]) != 2 for i in range(len(sequences)) if y_splice[i] == 2)
print(f"number of different 9-base windows around the {len(donor_windows)} donor sites: {len(set(donor_windows))}")
print(f"'neither' sequences with GT or AG at the center: {decoys} of {(y_splice == 2).sum().item()}")

<div class="alert alert-success">

<b>EXERCISE 5: a CNN for splice sites</b>

a) Complete `SpliceCNN` below: an `nn.Conv1d` with 4 input channels, 16 kernels of width 9 and padding 4, a ReLU, `nn.MaxPool1d(4)`, `nn.Flatten()`, and an `nn.Linear` to the 3 classes. Work out the number of inputs of the linear layer.

b) Train a `SpliceCNN` on `splice_train_loader` with Adam (learning rate 0.001), until it starts to overfit: use `fit_early_stopping` with at most 100 epochs and patience 5, with `splice_val_loader` as validation set. It returns the history of the training: plot the learning curves with `plot_history`, and print the validation accuracy of the restored model with `evaluate`.

c) <b>THINK:</b> The simple rule already reaches about 89%. Why can the CNN do better? Look at the output of the baseline cell.

d) Make a new class `GlobalPoolCNN`: the same as `SpliceCNN`, but with **global max pooling** (`nn.AdaptiveMaxPool1d(1)`) instead of `nn.MaxPool1d(4)`. Global max pooling keeps only the maximum of every feature map, wherever it is in the sequence. (What does this mean for the number of inputs of the linear layer?) Train it in the same way as in b). <b>THINK:</b> Why does this model perform so much worse, even worse than the simple rule? When is global max pooling a good choice?

</div>

a)

In [ ]:
class SpliceCNN(nn.Module):
    def __init__(self):
        super().__init__()
        ######## YOUR CODE HERE #########
        self.conv = ...    # Conv1d: 4 input channels, 16 kernels of width 9, padding 4
        self.pool = ...    # MaxPool1d(4)
        self.output = ...  # Linear to the 3 classes: how many inputs?
        #################################

    def forward(self, x):
        ######## YOUR CODE HERE #########
        # conv -> ReLU -> pool -> flatten -> linear
        return ...
        #################################


print(SpliceCNN()(X_splice[:8]).shape)  # should be (8, 3)

b)

In [ ]:
######## YOUR CODE HERE #########
torch.manual_seed(0)
splice_cnn = ...  # a new SpliceCNN
optimizer = ...   # Adam with learning rate 0.001
history = ...     # fit_early_stopping: at most 100 epochs, patience 5

# plot the learning curves with plot_history, and print the validation accuracy of the restored model

#################################

c)

**Your answer:**

d)

In [ ]:
######## YOUR CODE HERE #########
class GlobalPoolCNN(nn.Module):
    # the same as SpliceCNN, but with nn.AdaptiveMaxPool1d(1) instead of nn.MaxPool1d(4)
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return ...


# train a GlobalPoolCNN in the same way as in b), and print its validation accuracy

#################################

**Your answer:**

## 6 What did the network learn?

For the donor and the acceptor class, the cell below finds the most important kernel: the one with the largest weight in the output layer for that class. It shows the weights of the kernel as a heatmap (rows: A, C, G, T), with the **consensus** (the base with the largest weight at every position), and below it the weights of the output layer for that kernel at every position in the sequence: *where* the network uses the kernel.

In [ ]:
conv_weights = splice_cnn.conv.weight.detach()                     # (16 kernels, 4 bases, 9 positions)
output_weights = splice_cnn.output.weight.detach().view(3, 16, 15)  # (3 classes, 16 kernels, 15 pooled positions)

fig, axes = plt.subplots(2, 2, figsize=(14, 5), gridspec_kw={"height_ratios": [1, 1]})
for k in range(2):  # donor, acceptor
    best_kernel = output_weights[k].max(dim=1).values.argmax().item()
    weights = conv_weights[best_kernel]
    consensus = "".join("ACGT"[i] for i in weights.argmax(dim=0))
    
    ax = axes[0, k]
    ax.imshow(weights, cmap="RdBu_r", vmin=-weights.abs().max(), vmax=weights.abs().max())
    ax.set_yticks(range(4), list("ACGT"))
    ax.set_xticks(range(9), list(consensus))
    ax.set_title(f"{class_names[k]}: kernel {best_kernel}")

    ax = axes[1, k]
    ax.bar(np.arange(15) * 4 + 1.5, output_weights[k, best_kernel], width=3)
    ax.axvline(29.5, color="black", linestyle="--", label="center of the sequence")
    ax.set_xlabel("position in the sequence")
    ax.set_ylabel("output weight")
    ax.legend()
plt.tight_layout()
plt.show()

<div class="alert alert-success">

<b>EXERCISE 6 (THINK): interpreting the kernels</b>

Look at the consensus of the two kernels. Do you recognize the GT of the donor site and the AG of the acceptor site, and which bases does the network use around them? Where in the sequence does the network use these kernels?

</div>

**Your answer:**